In [ ]:
import scanpy as sc
import squidpy as sq
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import anndata as ad
from scipy import sparse
import anndata as ad
import json

In [1]:
%cd './src'

import importlib
import ssgsea
import ssgsea_plots
import multimodal

importlib.reload(ssgsea)
importlib.reload(ssgsea_plots)
importlib.reload(multimodal)

%cd ..

/home/lucia/repos/Thesis_final/src


/home/lucia/miniforge3/envs/masters_env_minimal/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/home/lucia/miniforge3/envs/masters_env_minimal/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/home/lucia/miniforge3/envs/masters_env_minimal/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead

/home/lucia/repos/Thesis_final


In [ ]:
# load the preprocessed data
adata = ad.read_h5ad("BRCA_preprocessed.h5ad", backed=None)  
library_id='Visium_Human_Breast_Cancer'
adata

In [ ]:
# run the image segmentation for all values of sigma, save in a new anndata object
adata_img = multimodal.run_img_segmentation(
                adata = adata,
                output_dir = "segmentation_results",
                sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
                superpixel = "slic",
                res = "hires"
                )

In [ ]:
# load the anndata object with the image segmentation results
adata_img = ad.read_h5ad("segmentation_results/adata_img_segmentation_slic.h5ad")

In [ ]:
# run ssGSEA if not done already
ssgsea = ssgsea.ssGSEA()
results = ssgsea.compute_ssgsea(adata, 'h.all.v2026.1.Hs.json', layer=None)
ssgsea_df = pd.DataFrame.from_dict(results)

In [ ]:
# load precomputed ssGSEA results
ssgsea_df = pd.read_csv('NES.csv', index_col=0)

In [ ]:
# run multimodal pipeline for EMT gene set

multimodal.multimodal_distance(    
    adata_img_segmentation = adata_img,
    adata_gene_expr = adata,
    ssgsea_df = ssgsea_df[['HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION']],
    modality_combinations = [False, True], # how I want to combine gexpr and gsets
    distance_gset = 'euclidean',
    superpixel = 'slic',
    sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
    weight_combinations = [(1,1), (0.4, 0.6)],
    cluster = True,
    linkage = 'complete',
    k = [8, 16, 20, 32],
    output_dir = "multimodal_results_EMT"
    )

In [ ]:
# plot clustering outputs for EMT

results_adata_path = Path('multimodal_results_EMT/')
dirs = list(results_adata_path.glob('*'))
for d in dirs:
    inp = str(d)
    out = str(d)+'/plots'
    print(inp, out)
    ssgsea_plots.show_results(ssgsea_df[['HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION']], 
                              gene_set='HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION', 
                              results_adata_dir=inp, output_dir=out)

In [ ]:
# run multimodal pipeline for ER late response gene set

multimodal.multimodal_distance(    
    adata_img_segmentation = adata_img,
    adata_gene_expr = adata,
    ssgsea_df = ssgsea_df[['HALLMARK_ESTROGEN_RESPONSE_LATE']],
    modality_combinations = [False, True], # how I want to combine gexpr and gsets
    distance_gset = 'euclidean',
    superpixel = 'slic',
    sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
    weight_combinations = [(1,1), (0.4, 0.6)],
    cluster = True,
    linkage = 'complete',
    k = [8, 16, 20, 32],
    output_dir = "multimodal_results_ER_late"
    )

In [ ]:
# plot clustering outputs for ER late response

results_adata_path = Path('multimodal_results_ER_late/')
dirs = list(results_adata_path.glob('*'))
for d in dirs:
    inp = str(d)
    out = str(d)+'/plots'
    print(inp, out)
    ssgsea_plots.show_results(ssgsea_df[['HALLMARK_ESTROGEN_RESPONSE_LATE']], 
                              gene_set='HALLMARK_ESTROGEN_RESPONSE_LATE', 
                              results_adata_dir=inp, output_dir=out)

In [ ]:
# run multimodal pipeline for MYC targets V1 gene set

multimodal.multimodal_distance(    
    adata_img_segmentation = adata_img,
    adata_gene_expr = adata,
    ssgsea_df = ssgsea_df[['HALLMARK_MYC_TARGETS_V1']],
    modality_combinations = [False, True], # how I want to combine gexpr and gsets
    distance_gset = 'euclidean',
    superpixel = 'slic',
    sigmas = [0, 0.5, 1, 2, 4, 6, 8, 10],
    weight_combinations = [(1,1), (0.4, 0.6)],
    cluster = True,
    linkage = 'complete',
    k = [8, 16, 20, 32],
    output_dir = "multimodal_results_MYC_V1"
    )

In [ ]:
# plot clustering outputs for MYC targets V1

results_adata_path = Path('multimodal_results_MYC_V1/')
dirs = list(results_adata_path.glob('*'))
for d in dirs:
    inp = str(d)
    out = str(d)+'/plots'
    print(inp, out)
    ssgsea_plots.show_results(ssgsea_df[['HALLMARK_MYC_TARGETS_V1']], 
                              gene_set='HALLMARK_MYC_TARGETS_V1', 
                              results_adata_dir=inp, output_dir=out)